# 11 Bridge to Rust Objects

Convert widget/session state into Rust-backed objects.

In [ ]:
import math

from nbplay import SamplerWidget, SequencerWidget, Session, SynthWidget

# ── set up sound sources ────────────────────────────────────────────────
lead = SynthWidget(oscillator_type="saw", frequency=660.0, amplitude=0.55)
bass = SynthWidget(oscillator_type="square", frequency=110.0, amplitude=0.65)

sampler = SamplerWidget(
    attack=0.005, decay=0.08, sustain=0.15, release=0.12, max_voices=6,
)
sample_rate = 44100
frame_count = int(sample_rate * 0.18)
sample_data = [
    math.sin(2 * math.pi * 880 * index / sample_rate) * (1 - index / frame_count)
    for index in range(frame_count)
]
sampler.load_sample(sample_data, sample_rate=sample_rate, root_note=60, name="Notebook Blip")

# ── sequencers ──────────────────────────────────────────────────────────
lead_seq = SequencerWidget(length=8, bpm=118.0)
for step_index, note in enumerate([72, 76, 79, 83, 79, 76, 74, 71]):
    lead_seq.set_step(step_index, note=note, velocity=102, active=True)

bass_seq = SequencerWidget(length=8, bpm=118.0)
for step_index, velocity in enumerate([118, 64, 88, 64, 110, 64, 92, 64]):
    bass_seq.set_step(step_index, note=48, velocity=velocity, active=True)

drum_seq = SequencerWidget(length=8, bpm=118.0)
for step_index in [0, 3, 4, 7]:
    drum_seq.set_step(step_index, note=60, velocity=112, active=True)

# ── session ─────────────────────────────────────────────────────────────
session = Session(bpm=118.0, time_signature=(4, 4))
session.add_track("Lead", lead_seq, lead)
session.add_track("Bass", bass_seq, bass)
session.add_track("Drums", drum_seq, sampler)

# ── convert to Rust types ──────────────────────────────────────────────
rust_mixer = session.mixer.to_mixer()
lead_pattern = lead_seq.to_pattern()
bass_pattern = bass_seq.to_pattern()
drum_pattern = drum_seq.to_pattern()
rust_sampler = sampler.to_sampler()

print(f"Mixer channels: {len(rust_mixer)}")
print(f"Lead pattern length: {len(lead_pattern)}")
print(f"Bass pattern length: {len(bass_pattern)}")
print(f"Drum pattern length: {len(drum_pattern)}")
print(f"Sampler active voices: {rust_sampler.active_voice_count()}")